In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [7]:
df = pd.read_csv("housing.csv")

In [8]:
print("First five rows:")
display(df.head())

print("Dataset shape:", df.shape)


First five rows:


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


Dataset shape: (20640, 10)


In [9]:
print("Missing values in each column:")
print(df.isnull().sum())

df["total_bedrooms"] = df["total_bedrooms"].fillna(df["total_bedrooms"].median())

print("Missing values in each column:")
print(df.isnull().sum())

Missing values in each column:
longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64
Missing values in each column:
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
ocean_proximity       0
dtype: int64


In [11]:
print("\n--- Duplicate Rows ---")
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

df = df.drop_duplicates()


--- Duplicate Rows ---
Number of duplicate rows: 0


In [12]:
print("\n--- Outlier Detection and Handling ---")

numeric_columns = df.select_dtypes(include=np.number).columns
outlier_summary = {}

for column in numeric_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = (df[column] < lower_bound) | (df[column] > upper_bound)
    outlier_count = outliers.sum()
    outlier_summary[column] = outlier_count

    print(f"{column}: {outlier_count} outliers detected")

    df[column] = df[column].clip(lower=lower_bound, upper=upper_bound)



--- Outlier Detection and Handling ---
longitude: 0 outliers detected
latitude: 0 outliers detected
housing_median_age: 0 outliers detected
total_rooms: 1287 outliers detected
total_bedrooms: 1306 outliers detected
population: 1196 outliers detected
households: 1220 outliers detected
median_income: 681 outliers detected
median_house_value: 1071 outliers detected



In [13]:
X = df.drop("median_house_value", axis=1)
y = df["median_house_value"]

X = pd.get_dummies(X, columns=["ocean_proximity"], drop_first=True)

print("Input data:")
display(X.head())

print("Output data:")
display(y.head())

Input data:


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,-122.23,37.88,41.0,880.000,129.0,322.0,126.0,8.013025,False,False,True,False
1,-122.22,37.86,21.0,5698.375,1106.0,2401.0,1092.5,8.013025,False,False,True,False
2,-122.24,37.85,52.0,1467.000,190.0,496.0,177.0,7.257400,False,False,True,False
3,-122.25,37.85,52.0,1274.000,235.0,558.0,219.0,5.643100,False,False,True,False
4,-122.25,37.85,52.0,1627.000,280.0,565.0,259.0,3.846200,False,False,True,False


Output data:


0    452600.0
1    358500.0
2    352100.0
3    341300.0
4    342200.0
Name: median_house_value, dtype: float64

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (16512, 12)
Testing data: (4128, 12)


In [16]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mse1 = mean_squared_error(y_test, y_pred)
mae1 = mean_absolute_error(y_test, y_pred)
r2_1 = r2_score(y_test, y_pred)

print("Linear Regression Results")
print("MSE:", mse1)
print("MAE:", mae1)
print("R2 Score:", r2_1)


Linear Regression Results
MSE: 4678607409.813624
MAE: 50416.4306056431
R2 Score: 0.6299612712831428


In [17]:
poly = PolynomialFeatures(degree=2)

X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)

y_poly_pred = poly_model.predict(X_test_poly)

mse2 = mean_squared_error(y_test, y_poly_pred)
mae2 = mean_absolute_error(y_test, y_poly_pred)
r2_2 = r2_score(y_test, y_poly_pred)

print("Polynomial Regression Results")
print("MSE:", mse2)
print("MAE:", mae2)
print("R2 Score:", r2_2)

Polynomial Regression Results
MSE: 5610851554.7490425
MAE: 48130.13519307244
R2 Score: 0.5562285538249414


In [18]:
if r2_1 > r2_2:
    print("Linear Regression performs better based on R2 Score.")
elif r2_2 > r2_1:
    print("Polynomial Regression performs better based on R2 Score.")
else:
    print("Both models have the same R2 Score.")

Linear Regression performs better based on R2 Score.
